# Building a Raw ReAct Loop from Scratch with OpenAI

This tutorial demonstrates how to implement the **ReAct (Reasoning + Acting)** pattern manually using the OpenAI API, structured JSON outputs, and custom tool routing.

### The ReAct Mechanics
* **Thought (Reasoning):** The model analyzes the request and determines what action to take.
* **Action (Acting):** The model returns a structured tool call in JSON format.
* **Observation:** Your code executes the local Python function and feeds the output back into the message history.
* **Final Answer:** The loop repeats until the model chooses the `final_answer` action.

--- 
## Step 1: Install Dependencies
We install `openai` and `python-dotenv` for managing the API calls and environment keys.

In [ ]:
!pip install -q openai python-dotenv

--- 
## Step 2: Set Up Environment Variables & Client
Securely load your OpenAI API key and instantiate the OpenAI client.

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

if "OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"]:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

client = OpenAI()

--- 
## Step 3: Define System Prompt & Local Tool
The system prompt instructs the model to return strict JSON describing its thought process and tool execution choice.

In [ ]:
import json

SYSTEM_PROMPT = """Solve the user's question using a Thought -> Action -> Observation loop.

TOOL:
- get_weather: {"city": "string"} -> current weather for a city.

Respond with exactly one JSON object per turn:
{"thought": "...", "action": {"name": "get_weather", "parameters": {"city": "..."}}}
Once you know the answer, respond with:
{"thought": "...", "action": {"name": "final_answer", "parameters": {"response": "..."}}}
"""

WEATHER_BY_CITY = {
    "tokyo": "18°C, Rainy",
    "bangalore": "22°C, Sunny",
    "london": "14°C, Overcast",
    "new york": "26°C, Humid",
}

def get_weather(city: str) -> str:
    return WEATHER_BY_CITY.get(city.lower().strip(), "20°C, Partly Cloudy")

--- 
## Step 4: Implement the Execution Loop
This loop queries `gpt-4o-mini`, parses the JSON output, executes the corresponding Python code, and feeds back observations.

In [ ]:
def react_loop(user_query: str, max_steps: int = 5):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for step in range(max_steps):
        print(f"--- Step {step + 1} ---")
        
        reply = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            response_format={"type": "json_object"},
        ).choices[0].message.content

        data = json.loads(reply)
        print("Model Output:", json.dumps(data, indent=2))
        messages.append({"role": "assistant", "content": reply})

        action = data["action"]

        if action["name"] == "final_answer":
            print(f"\nFinal Answer: {action['parameters']['response']}")
            return

        # Execute Action
        observation = get_weather(**action["parameters"])
        print(f"Execution Observation: {observation}\n")

        # Pass Observation back to the loop
        messages.append({"role": "user", "content": f"Observation: {observation}"})

--- 
## Step 5: Execute Query
Test the custom ReAct agent with a query that requires multiple tool executions.

In [ ]:
react_loop("What's the weather like in Tokyo and Bangalore?")